# Evaluate PANGAEA descriptive metadata

Faithfulness and similarity are the mean ± standard error over the 75 PANGAEA datasets, in percent. Faithfulness is the rubric score rescaled from 1–5 to 0–1. Similarity is factual-correctness precision against the reference description. A dataset with no successful description scores zero.

M³, AutoDDG, DataAtlas, Single LLM, and ReAct are read from `outputs/pangaea/df_scores_pangaea_descriptive_with_baselines.csv`. M³ fast is read from the judge checkpoints in `outputs/m3_fast_vs_base/descriptive_judge*`. The best model for a method is the highest mean of faithfulness and similarity.


In [1]:
import asyncio
import hashlib
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import FactualCorrectness, RubricsScore

REPO = Path('/home/com3dian/Github/process_data_metadata_agent')
PRED = Path('/home/com3dian/Github/metadata_agent/outputs/m3_fast_pangaea_descriptive_Mistral-Small-3.2-24B-Instruct-2506/m3/pangaea')
ANNOT = REPO / 'outputs' / 'pangaea' / 'sample_20260629_163144_filtered_top5_per_topic.csv'
DATA = REPO / 'outputs' / 'pangaea' / 'pangaea_datasets'
JUDGE_DIR = REPO / 'outputs' / 'm3_fast_vs_base' / 'descriptive_judge_mistral_small'
JUDGE_MODEL = 'mistralai/Devstral-Small-2-24B-Instruct-2512'
COL_NAMES = ['name', 'description', 'keywords']

load_dotenv(REPO / '.env', override=True)
SURF_API_KEY = os.getenv('SURF_API_KEY')
SURF_API_BASE = os.getenv('SURF_API_BASE') or 'https://willma.surf.nl/api/v0/'
if not SURF_API_KEY:
    raise RuntimeError(f'SURF_API_KEY is missing from {REPO / ".env"}')
JUDGE_DIR.mkdir(parents=True, exist_ok=True)

judge_llm = LangchainLLMWrapper(ChatOpenAI(
    model=JUDGE_MODEL,
    api_key=SURF_API_KEY,
    base_url=SURF_API_BASE,
    default_headers={'X-API-KEY': SURF_API_KEY},
    temperature=0,
    max_tokens=2048,
    timeout=600,
    max_retries=4,
))
grounding_scorer = RubricsScore(
    rubrics={
        'score1_description': 'Makes claims that contradict the data or reference fields/values that do not exist in it.',
        'score2_description': 'Several claims are unsupported by the data or appear invented.',
        'score3_description': 'Mostly supportable, but contains at least one unsupported or speculative claim.',
        'score4_description': 'Every claim is supportable from the data; some claims are vague or generic.',
        'score5_description': 'Every claim is directly supported by the data and is specific to it.',
    },
    llm=judge_llm,
)
similarity_scorer = FactualCorrectness(llm=judge_llm, mode='precision', atomicity='low')


def metadata_sha256(metadata):
    return hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()


def serialize_dataset(frame):
    schema = '\n'.join(f'- {column} ({dtype})' for column, dtype in frame.dtypes.items())
    sample = frame.head(5).to_markdown(index=False)
    return f'Columns:\n{schema}\n\nSample rows:\n{sample}\n\nRow count: {len(frame)}'


async def score_description(dataset_id, metadata, reference_description):
    data = pd.read_csv(DATA / f'{int(dataset_id)}.csv')
    context = serialize_dataset(data)
    predicted = {key: metadata.get(key) for key in COL_NAMES}
    predicted['keywords'] = str(predicted['keywords'])
    if any(not isinstance(predicted[key], str) or not predicted[key].strip() for key in COL_NAMES):
        raise ValueError(f'Missing descriptive field for dataset {dataset_id}')
    abstract = '\n\n'.join(f'{key}: {predicted[key]}' for key in COL_NAMES)
    grounding_sample = SingleTurnSample(
        user_input=(
            'A model generated metadata for a dataset from the data alone.\n\n'
            f'DATASET:\n{context}\n\n'
            'Judge only whether the TITLE, DESCRIPTION, and KEYWORDS below are grounded in this data — '
            'whether its claims could be inferred from the data. Do not reward or '
            'penalize style, completeness, or omissions.'
        ),
        response=abstract,
    )
    similarity_sample = SingleTurnSample(
        response=predicted['description'],
        reference=reference_description,
    )
    rating = await grounding_scorer.single_turn_ascore(grounding_sample)
    similarity = await similarity_scorer.single_turn_ascore(similarity_sample)
    return (float(rating) - 1) / 4, float(similarity)


async def score_mistral_small():
    annotations = pd.read_csv(ANNOT).set_index('id')
    pending = []
    for dataset_id in annotations.index.astype(int):
        record = json.loads((PRED / f'{dataset_id}.json').read_text(encoding='utf-8'))
        digest = metadata_sha256(record['metadata'])
        checkpoint = JUDGE_DIR / f'{dataset_id}.json'
        if checkpoint.is_file():
            saved = json.loads(checkpoint.read_text(encoding='utf-8'))
            if saved.get('status') == 'ok' and saved.get('metadata_sha256') == digest:
                continue
        pending.append((dataset_id, record, digest))
    print(f'{len(annotations) - len(pending)} already scored, {len(pending)} still to judge')
    for dataset_id, record, digest in pending:
        try:
            faithfulness, similarity = await score_description(
                dataset_id,
                record['metadata'],
                annotations.loc[dataset_id, 'description'],
            )
            if not (0 <= faithfulness <= 1 and 0 <= similarity <= 1):
                raise ValueError(f'Judge score outside 0-1: {faithfulness}, {similarity}')
            payload = {
                'dataset_id': int(dataset_id),
                'status': 'ok',
                'metadata_sha256': digest,
                'judge_model': JUDGE_MODEL,
                'faithfulness': faithfulness,
                'similarity': similarity,
            }
        except Exception as exc:
            payload = {
                'dataset_id': int(dataset_id),
                'status': 'judge_error',
                'error': f'{type(exc).__name__}: {exc}',
            }
            print(f'Row {dataset_id} failed: {payload["error"]}')
        checkpoint = JUDGE_DIR / f'{dataset_id}.json'
        temporary = checkpoint.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(payload), encoding='utf-8')
        temporary.replace(checkpoint)
    faith, sim = [], []
    for dataset_id in annotations.index.astype(int):
        saved = json.loads((JUDGE_DIR / f'{dataset_id}.json').read_text(encoding='utf-8'))
        if saved.get('status') != 'ok':
            faith.append(0.0)
            sim.append(0.0)
            continue
        faith.append(float(saved['faithfulness']))
        sim.append(float(saved['similarity']))
    faith, sim = pd.Series(faith), pd.Series(sim)
    print(
        f'M³ fast Mistral Small 3.2 24B: '
        f'faithfulness {100 * faith.mean():.1f} ± {100 * faith.sem():.1f}, '
        f'similarity {100 * sim.mean():.1f} ± {100 * sim.sem():.1f}'
    )

await score_mistral_small()


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
/tmp/ipykernel_217637/2697190919.py:12: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, RubricsScore
/tmp/ipykernel_217637/2697190919.py:12: DeprecationWarning: Importing RubricsScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collecti

8 already scored, 67 still to judge
M³ fast Mistral Small 3.2 24B: faithfulness 96.0 ± 1.5, similarity 41.5 ± 3.1


In [2]:
import json
from pathlib import Path

import pandas as pd

REPO = Path('/home/com3dian/Github/process_data_metadata_agent')
SCORES_PATH = REPO / 'outputs' / 'pangaea' / 'df_scores_pangaea_descriptive_with_baselines.csv'
FAST_ROOT = REPO / 'outputs' / 'm3_fast_vs_base'
OUT = REPO / 'outputs' / 'pangaea'

DISPLAY_NAMES = {
    'llama3_70B': 'Llama3 70B',
    'default-text-large': 'Llama3 70B',
    'Qwen_3.6_35B': 'Qwen 3.6 35B',
    'Qwen_Qwen3.6-35B-A3B-FP8': 'Qwen 3.6 35B',
    'Qwen3.5-122B': 'Qwen 3.5 122B',
    'Sehyo_Qwen3.5-122B-A10B-NVFP4': 'Qwen 3.5 122B',
    'mistralai_Devstral-Small-2-24B-Instruct-2512': 'Mistral Devstral 2 24B',
    'mistralai_Mistral-Small-3.2-24B-Instruct-2506': 'Mistral Small 3.2 24B',
}
MODEL_ORDER = [
    'Qwen 3.6 35B',
    'Qwen 3.5 122B',
    'Llama3 70B',
    'Mistral Devstral 2 24B',
    'Mistral Small 3.2 24B',
]
BASELINE_PREFIXES = ('AutoDDG_', 'DataAtlas_', 'SingleLLM_', 'ReAct_')
FAST_RUNS = [
    ('Llama3 70B', FAST_ROOT / 'descriptive_judge'),
    ('Qwen 3.6 35B', FAST_ROOT / 'descriptive_judge_qwen'),
    ('Qwen 3.5 122B', FAST_ROOT / 'descriptive_judge_qwen35'),
    ('Mistral Devstral 2 24B', FAST_ROOT / 'descriptive_judge_devstral'),
    ('Mistral Small 3.2 24B', FAST_ROOT / 'descriptive_judge_mistral_small'),
]


def percent_summary(series):
    return 100 * float(series.mean()), 100 * float(series.sem())


def score_text(mean, sem):
    return f'{round(mean, 1)} ± {round(sem, 1)}'


def result_row(method, model, faithfulness, similarity):
    faith_mean, faith_sem = percent_summary(faithfulness)
    sim_mean, sim_sem = percent_summary(similarity)
    return {
        'Method': method,
        'Model': model,
        'Faithfulness [%]': score_text(faith_mean, faith_sem),
        'Similarity [%]': score_text(sim_mean, sim_sem),
        'Faith mean': faith_mean,
        'Sim mean': sim_mean,
        'Average': (faith_mean + sim_mean) / 2,
    }


def rows_from_saved_scores(df_scores, method, prefix):
    rows = []
    for column in df_scores.columns:
        if not column.startswith('faith_'):
            continue
        key = column.removeprefix('faith_')
        if prefix:
            if not key.startswith(prefix):
                continue
            model_key = key.removeprefix(prefix)
        elif key.startswith(BASELINE_PREFIXES):
            continue
        else:
            model_key = key
        rows.append(result_row(
            method,
            DISPLAY_NAMES.get(model_key, model_key),
            df_scores[column],
            df_scores[f'sim_{key}'],
        ))
    return rows


def fast_scores(dataset_ids, folder):
    faithfulness = []
    similarity = []
    for dataset_id in dataset_ids:
        path = folder / f'{int(dataset_id)}.json'
        if not path.is_file():
            faithfulness.append(0.0)
            similarity.append(0.0)
            continue
        saved = json.loads(path.read_text(encoding='utf-8'))
        if saved.get('status') != 'ok':
            faithfulness.append(0.0)
            similarity.append(0.0)
            continue
        faithfulness.append(float(saved['faithfulness']))
        similarity.append(float(saved['similarity']))
    return pd.Series(faithfulness), pd.Series(similarity)


def sort_models(frame):
    ordered = frame.copy()
    ordered['Model'] = pd.Categorical(ordered['Model'], MODEL_ORDER, ordered=True)
    return ordered.sort_values('Model').reset_index(drop=True)


def method_latex(df, caption, label):
    shown = df[['Method', 'Model', 'Faithfulness [%]', 'Similarity [%]']].copy()
    shown['Method'] = shown['Method'].replace({
        'M³ fast': 'M3FASTPLACEHOLDER',
        'M³': 'M3PLACEHOLDER',
    })
    grouped = shown.set_index(['Method', 'Model'])
    latex = grouped.to_latex(
        index=True,
        escape=True,
        multirow=True,
        column_format='llcc',
        caption=caption,
        label=label,
    )
    header = (
        ' &  & Faithfulness [\\%] & Similarity [\\%] \\\\\n'
        'Method & Model &  &  \\\\\n'
    )
    latex = latex.replace(
        header,
        'Method & Model & Faithfulness [\\%] & Similarity [\\%] \\\\\n',
    )
    latex = latex.replace(r'\cline{1-4}', r'\midrule')
    latex = latex.replace('\\midrule\n\\bottomrule', '\\bottomrule')
    latex = latex.replace('M3FASTPLACEHOLDER', r'M$^3$ fast')
    latex = latex.replace('M3PLACEHOLDER', r'M$^3$')
    latex = latex.replace('±', r'$\pm$')
    return latex


df_scores = pd.read_csv(SCORES_PATH)
rows = []
for method, prefix in [
    ('M³', ''),
    ('AutoDDG', 'AutoDDG_'),
    ('DataAtlas', 'DataAtlas_'),
    ('Single LLM', 'SingleLLM_'),
    ('ReAct', 'ReAct_'),
]:
    rows.extend(rows_from_saved_scores(df_scores, method, prefix))
for model, folder in FAST_RUNS:
    faithfulness, similarity = fast_scores(df_scores['id'], folder)
    rows.append(result_row('M³ fast', model, faithfulness, similarity))

all_models = pd.concat(
    [sort_models(frame) for _, frame in pd.DataFrame(rows).groupby('Method', sort=False)],
    ignore_index=True,
)
best_rows = []
for method, group in all_models.groupby('Method', sort=False):
    best_rows.append(group.sort_values('Average', ascending=False, kind='mergesort').iloc[0])
best_models = pd.DataFrame(best_rows).reset_index(drop=True)
best_models['Model'] = best_models['Model'].astype(str)

all_caption = (
    r'Faithfulness and similarity for every model of M$^3$, M$^3$ fast, AutoDDG, '
    r'DataAtlas, Single LLM, and ReAct on PANGAEA. Values are mean $\pm$ standard error, in percent.'
)
best_caption = (
    r'Best model per method on PANGAEA descriptive metadata. '
    r'The best model is the highest mean of faithfulness and similarity. '
    r'Values are mean $\pm$ standard error, in percent.'
)
all_latex = method_latex(all_models, all_caption, 'tab:all_models_descriptive')
best_shown = best_models.rename(columns={'Model': 'Best model'})
best_latex = best_shown[['Method', 'Best model', 'Faithfulness [%]', 'Similarity [%]']].copy()
best_latex['Method'] = best_latex['Method'].replace({
    'M³ fast': 'M3FASTPLACEHOLDER',
    'M³': 'M3PLACEHOLDER',
})
best_latex = best_latex.to_latex(
    index=False,
    escape=True,
    column_format='llcc',
    caption=best_caption,
    label='tab:best_method_descriptive',
)
best_latex = best_latex.replace('M3FASTPLACEHOLDER', r'M$^3$ fast')
best_latex = best_latex.replace('M3PLACEHOLDER', r'M$^3$')
best_latex = best_latex.replace('±', r'$\pm$')

all_path = OUT / 'df_results_all_methods_descriptive.tex'
best_path = OUT / 'df_results_best_method_descriptive.tex'
all_path.write_text(all_latex, encoding='utf-8')
best_path.write_text(best_latex, encoding='utf-8')
print(all_latex)
print(best_latex)
print(f'Saved {all_path}')
print(f'Saved {best_path}')
all_models[['Method', 'Model', 'Faithfulness [%]', 'Similarity [%]', 'Average']].round({'Average': 2})


\begin{table}
\caption{Faithfulness and similarity for every model of M$^3$, M$^3$ fast, AutoDDG, DataAtlas, Single LLM, and ReAct on PANGAEA. Values are mean $\pm$ standard error, in percent.}
\label{tab:all_models_descriptive}
\begin{tabular}{llcc}
\toprule
Method & Model & Faithfulness [\%] & Similarity [\%] \\
\midrule
\multirow[t]{5}{*}{M$^3$} & Qwen 3.6 35B & 95.3 $\pm$ 2.1 & 45.1 $\pm$ 3.6 \\
 & Qwen 3.5 122B & 98.3 $\pm$ 1.0 & 33.1 $\pm$ 2.8 \\
 & Llama3 70B & 94.7 $\pm$ 1.7 & 56.0 $\pm$ 3.4 \\
 & Mistral Devstral 2 24B & 96.7 $\pm$ 1.5 & 43.9 $\pm$ 3.1 \\
 & Mistral Small 3.2 24B & 25.0 $\pm$ 4.7 & 12.7 $\pm$ 2.7 \\
\midrule
\multirow[t]{5}{*}{AutoDDG} & Qwen 3.6 35B & 80.7 $\pm$ 3.1 & 43.2 $\pm$ 2.6 \\
 & Qwen 3.5 122B & 81.3 $\pm$ 3.7 & 38.0 $\pm$ 2.5 \\
 & Llama3 70B & 68.3 $\pm$ 4.4 & 17.6 $\pm$ 1.7 \\
 & Mistral Devstral 2 24B & 82.0 $\pm$ 3.2 & 41.9 $\pm$ 2.8 \\
 & Mistral Small 3.2 24B & 82.7 $\pm$ 3.3 & 44.4 $\pm$ 2.8 \\
\midrule
\multirow[t]{5}{*}{DataAtlas} & Qwen 3.

,Method,Model,Faithfulness [%],Similarity [%],Average
0,M³,Qwen 3.6 35B,95.3 ± 2.1,45.1 ± 3.6,70.20
1,M³,Qwen 3.5 122B,98.3 ± 1.0,33.1 ± 2.8,65.70
2,M³,Llama3 70B,94.7 ± 1.7,56.0 ± 3.4,75.35
3,M³,Mistral Devstral 2 24B,96.7 ± 1.5,43.9 ± 3.1,70.30
4,M³,Mistral Small 3.2 24B,25.0 ± 4.7,12.7 ± 2.7,18.85
5,AutoDDG,Qwen 3.6 35B,80.7 ± 3.1,43.2 ± 2.6,61.93
6,AutoDDG,Qwen 3.5 122B,81.3 ± 3.7,38.0 ± 2.5,59.65
7,AutoDDG,Llama3 70B,68.3 ± 4.4,17.6 ± 1.7,42.96
8,AutoDDG,Mistral Devstral 2 24B,82.0 ± 3.2,41.9 ± 2.8,61.93
9,AutoDDG,Mistral Small 3.2 24B,82.7 ± 3.3,44.4 ± 2.8,63.55
